# Building conversational applications with CrewAI

In this notebook, you learn how to use CrewAI to create collaborative AI agents that can work together to solve complex tasks.

This demonstrates the same functionality as the Strands Agents example, but using CrewAI's multi-agent approach.

## Environment setup

In this task, you set up your environment and install the required packages for CrewAI.

In [1]:
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

model_id = "amazon.nova-lite-v1:0"

## Define Tools for CrewAI

In this task, we define the tools that our CrewAI agents will use.

In [2]:
@tool
def calculator_tool(oper1: int, oper2: int, operator: str) -> str:
    """
    Perform basic arithmetic operations on two numbers.
    
    Args:
        oper1: First operand
        oper2: Second operand  
        operator: Operation to perform (+, -, *, / or add, subtract, multiply, divide)
    
    Returns:
        Result of the calculation as a string
    """
    def add(a, b):
        return a + b

    def subtract(a, b):
        return a - b

    def multiply(a, b):
        return a * b

    def divide(a, b):
        if b == 0:
            return "Cannot divide by zero"
        else:
            return float(a / b)
    
    if operator == '+' or operator == 'add':
        return str(add(oper1, oper2))
    elif operator == '-' or operator == 'subtract':
        return str(subtract(oper1, oper2))
    elif operator == '*' or operator == 'multiply':
        return str(multiply(float(oper1), float(oper2)))
    elif operator == '/' or operator == 'divide':
        return str(divide(oper1, oper2))
    else:
        return "Invalid operator"

In [3]:
@tool
def search_tool(query: str) -> str:
    """
    Search with DuckDuckGo for information about a topic.
    
    Args:
        query: The search query
    
    Returns:
        DuckDuckGo search results as a string
    """
    tool = DuckDuckGoSearchRun()
    result = tool.invoke(query)
    return result

## Create CrewAI Agents and Tasks

Now we create specialized agents for different tasks and define how they work together.

In [4]:
# Create the Bedrock LLM
llm = LLM(
    model="bedrock/us.amazon.nova-lite-v1:0"
)

# Create agent
agent = Agent(
    role="Expert in searching and calculations",
    goal="Use available tools to find Find accurate information on any topic using tools",
    backstory="You are a helpful assistant that can perform calculations and search for information on DuckDuckGo. Use the available tools when needed to provide accurate and helpful responses.",
    tools=[search_tool, calculator_tool],
    llm=llm,
    verbose=True
)

## Test the CrewAI System

Now let's test our CrewAI system with the same queries used in the Strands example.

In [5]:
def run_crew_task(user_query: str):
    """Run a CrewAI task based on user query"""
    
    # Create task
    task = Task(
        description=f"Answer this query: {user_query}",
        expected_output="A comprehensive answer to the user's query",
        agent=agent
    )
    
    # Create crew
    crew = Crew(
        agents=[agent],
        tasks=[task],
        verbose=True
    )
    
    # Execute
    result = crew.kickoff()
    return result

In [6]:
# Test with the same complex query from Strands example
query = "What is Amazon SageMaker? What is launch year multiplied by 2"

print(f"Query: {query}\n")
print("CrewAI Response:")
print("=" * 50)

response = run_crew_task(query)
print(response)

Query: What is Amazon SageMaker? What is launch year multiplied by 2

CrewAI Response:


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5e909c1c-65f0-4193-9726-5dbd63099184                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer this query: What is Amazon SageMaker? What is launch year multiplied by 2                         │
│  ID: 4a4c5bdf-b42a-44eb-b309-33e4313821b6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert in searching and calculations                                                                    │
│                                                                                                                 │
│  Task: Answer this query: What is Amazon SageMaker? What is launch year multiplied by 2                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_tool                                                                                              │
│  Args: {'query': 'What is Amazon SageMaker?'}                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert in searching and calculations                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  <thinking>The query has been answered. I have the information about Amazon SageMaker and the calculation       │
│  result.</thinking>                                                                                             │
│  <Final Answer> Amazon SageMaker is a fully managed service designed to simplify the process of building,       │
│  training, and deploying machine learning (ML) models. It is a cloud-based machine learning platform that       │
│  allows the creation, training, and deployment by developers of machine-learning (ML) models on the cloud. The  │
│  launch year of Amazon SageMaker is 2017, and when multiplied by 2, the result is 4034.</Final Answer>          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer this query: What is Amazon SageMaker? What is launch year multiplied by 2                         │
│  Agent: Expert in searching and calculations                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5e909c1c-65f0-4193-9726-5dbd63099184                                                                       │
│  Final Output: <thinking>The query has been answered. I have the information about Amazon SageMaker and the     │
│  calculation result.</thinking>                                                                                 │
│  <Final Answer> Amazon SageMaker is a fully managed service designed to simplify the process of building,       │
│  training, and deploying machine learning (ML) models. It is a cloud-based machine learning platform that       │
│  allows the creation, training, and deployment by developers of machine-learning (ML) models on the cloud. The  │
│  launch year of Amazon SageMaker is 2017, and when multiplied by 2, the result is 4034.</Final Answer>          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

<thinking>The query has been answered. I have the information about Amazon SageMaker and the calculation result.</thinking>
<Final Answer> Amazon SageMaker is a fully managed service designed to simplify the process of building, training, and deploying machine learning (ML) models. It is a cloud-based machine learning platform that allows the creation, training, and deployment by developers of machine-learning (ML) models on the cloud. The launch year of Amazon SageMaker is 2017, and when multiplied by 2, the result is 4034.</Final Answer>


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
# Test with calculation-only query
calc_query = "What is 156 multiplied by 23?"
print(f"Query: {calc_query}")
print(f"Response: {run_crew_task(calc_query)}")
print("\n" + "="*50 + "\n")

Query: What is 156 multiplied by 23?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7b415edf-bd78-451a-a6e0-074b437984d9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer this query: What is 156 multiplied by 23?                                                         │
│  ID: 8a000754-f989-48c9-a7c2-455c3d9be2e1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert in searching and calculations                                                                    │
│                                                                                                                 │
│  Task: Answer this query: What is 156 multiplied by 23?                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool calculator_tool executed with result: 3588.0...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert in searching and calculations                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  <thinking>The query has been answered. I have the calculation result.</thinking>                               │
│  <Final Answer> The result of multiplying 156 by 23 is 3588.</Final Answer>                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer this query: What is 156 multiplied by 23?                                                         │
│  Agent: Expert in searching and calculations                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7b415edf-bd78-451a-a6e0-074b437984d9                                                                       │
│  Final Output: <thinking>The query has been answered. I have the calculation result.</thinking>                 │
│  <Final Answer> The result of multiplying 156 by 23 is 3588.</Final Answer>                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Response: <thinking>The query has been answered. I have the calculation result.</thinking>
<Final Answer> The result of multiplying 156 by 23 is 3588.</Final Answer>




╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# Test with search-only query
search_query = "Tell me about artificial intelligence"
print(f"Query: {search_query}")
print(f"Response: {run_crew_task(search_query)}")
print("\n" + "="*50 + "\n")